In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## Load, split and scale the California housing dataset

In [2]:
housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)

In [3]:
tf.random.set_seed(42)

# Create the normalization layer separately
norm_layer = tf.keras.layers.Normalization()

# Build the model using the normalization layer
model = tf.keras.Sequential([
    tf.keras.Input(shape=X_train.shape[1:]),
    norm_layer,  # use the same reference
    tf.keras.layers.Dense(50, activation="relu"),
    tf.keras.layers.Dense(50, activation="relu"),
    tf.keras.layers.Dense(50, activation="relu"),
    tf.keras.layers.Dense(1)
])

# Adapt the normalization layer
norm_layer.adapt(X_train)

# Compile and train
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss="mse", optimizer=optimizer, metrics=["RootMeanSquaredError"])

history = model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))

# Evaluate and predict
mse_test, rmse_test = model.evaluate(X_test, y_test)
X_new = X_test[:3]
y_pred = model.predict(X_new)

Epoch 1/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.2915 - loss: 1.8017 - val_RootMeanSquaredError: 0.8832 - val_loss: 0.7800
Epoch 2/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - RootMeanSquaredError: 0.6395 - loss: 0.4096 - val_RootMeanSquaredError: 0.8231 - val_loss: 0.6775
Epoch 3/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.6117 - loss: 0.3744 - val_RootMeanSquaredError: 1.1660 - val_loss: 1.3595
Epoch 4/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.5998 - loss: 0.3599 - val_RootMeanSquaredError: 0.8693 - val_loss: 0.7557
Epoch 5/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 0.5879 - loss: 0.3458 - val_RootMeanSquaredError: 0.6418 - val_loss: 0.4119
Epoch 6/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - RootMeanSquaredError: 0.5788 - loss: 0.3352 - val_RootMeanSquaredError: 0.6854 - val_loss: 0.4698
Epoch 7/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - RootMeanSquaredError: 0.5720 - los

In [4]:
rmse_test

0.5335909724235535

In [5]:
y_pred

array([[0.44698066],
       [1.2784756 ],
       [5.1053    ]], dtype=float32)

## Building Complex Models Using the Functional API

Not all NN models are simply sequential - some may have complex topologies, some may have multiple inputs and/or multiple outputs. For example, a Wide & Deep neural network (see [paper](https://research.google/pubs/wide-deep-learning-for-recommender-systems/)) connects all or part of the inputs directly to the output layer.

In [6]:
# Clear any existing Keras session to free memory and reset layer naming
tf.keras.backend.clear_session()

# Set random seed for reproducibility (e.g., same weight initialization and training behavior)
tf.random.set_seed(42)

In [7]:
normalization_layer = tf.keras.layers.Normalization()
hidden_layer1 = tf.keras.layers.Dense(30, activation="relu")
hidden_layer2 = tf.keras.layers.Dense(30, activation="relu")
concat_layer = tf.keras.layers.Concatenate()
output_layer = tf.keras.layers.Dense(1)

input_ = tf.keras.layers.Input(shape=X_train.shape[1:])
normalized = normalization_layer(input_)
hidden1 = hidden_layer1(normalized)
hidden2 = hidden_layer2(hidden1)
concat = concat_layer([normalized, hidden2])
output = output_layer(concat)

model = tf.keras.Model(inputs=[input_], outputs=[output])

In [8]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 8)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ normalization (Normalization) │ (None, 8)                 │              17 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 30)                │             270 │ normalization[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 30)                │             930 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate (Concatenate)     │ (None, 38)                │               0 │ normalization[0][0],       │
│                               │                           │                 │ dense_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 1)                 │              39 │ concatenate[0][0]          │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,256 (4.91 KB)

 Trainable params: 1,239 (4.84 KB)

 Non-trainable params: 17 (72.00 B)

In [9]:
# Adapt normalization before compiling the model
normalization_layer.adapt(X_train)

# Compile after the normalization layer is ready
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss="mse", optimizer=optimizer, metrics=["RootMeanSquaredError"])

# Train, evaluate, and predict
history = model.fit(X_train, y_train, epochs=20, validation_data=(X_valid, y_valid))
mse_test = model.evaluate(X_test, y_test)
y_pred = model.predict(X_new)

Epoch 1/20


C:\LLM Instruction Fine-Tuning\.venv\Lib\site-packages\keras\src\models\functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(None, 8))
  warnings.warn(msg)


363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - RootMeanSquaredError: 1.4514 - loss: 2.2591 - val_RootMeanSquaredError: 0.8631 - val_loss: 0.7449
Epoch 2/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - RootMeanSquaredError: 0.7261 - loss: 0.5289 - val_RootMeanSquaredError: 0.9257 - val_loss: 0.8569
Epoch 3/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - RootMeanSquaredError: 0.6467 - loss: 0.4186 - val_RootMeanSquaredError: 1.0524 - val_loss: 1.1076
Epoch 4/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - RootMeanSquaredError: 0.6249 - loss: 0.3908 - val_RootMeanSquaredError: 0.9403 - val_loss: 0.8842
Epoch 5/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - RootMeanSquaredError: 0.6127 - loss: 0.3756 - val_RootMeanSquaredError: 0.7653 - val_loss: 0.5857
Epoch 6/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - RootMeanSquaredError: 0.6031 - loss: 0.3640 - val_RootMeanSquaredError: 0.7844 - val_loss: 0.6153
Epoch 7/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - RootMeanSquaredError: 0.5958 - loss: 0.3552 -

C:\LLM Instruction Fine-Tuning\.venv\Lib\site-packages\keras\src\models\functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(3, 8))
  warnings.warn(msg)


What if you want to send different subsets of input features through the wide or deep paths? We will send 5 features (features 0 to 4), and 6 through the deep path (features 2 to 7). Note that 3 features will go through both (features 2, 3 and 4).

In [10]:
tf.random.set_seed(42)

In [11]:
input_wide = tf.keras.layers.Input(shape=[5])  # features 0 to 4
input_deep = tf.keras.layers.Input(shape=[6])  # features 2 to 7
norm_layer_wide = tf.keras.layers.Normalization()
norm_layer_deep = tf.keras.layers.Normalization()
norm_wide = norm_layer_wide(input_wide)
norm_deep = norm_layer_deep(input_deep)
hidden1 = tf.keras.layers.Dense(30, activation="relu")(norm_deep)
hidden2 = tf.keras.layers.Dense(30, activation="relu")(hidden1)
concat = tf.keras.layers.concatenate([norm_wide, hidden2])
output = tf.keras.layers.Dense(1)(concat)
model = tf.keras.Model(inputs=[input_wide, input_deep], outputs=[output])

In [12]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss="mse", optimizer=optimizer, metrics=["RootMeanSquaredError"])

X_train_wide, X_train_deep = X_train[:, :5], X_train[:, 2:]
X_valid_wide, X_valid_deep = X_valid[:, :5], X_valid[:, 2:]
X_test_wide, X_test_deep = X_test[:, :5], X_test[:, 2:]
X_new_wide, X_new_deep = X_test_wide[:3], X_test_deep[:3]

norm_layer_wide.adapt(X_train_wide)
norm_layer_deep.adapt(X_train_deep)
history = model.fit((X_train_wide, X_train_deep), y_train, epochs=20,
                    validation_data=((X_valid_wide, X_valid_deep), y_valid))
mse_test = model.evaluate((X_test_wide, X_test_deep), y_test)
y_pred = model.predict((X_new_wide, X_new_deep))

Epoch 1/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - RootMeanSquaredError: 1.5269 - loss: 2.4345 - val_RootMeanSquaredError: 0.8463 - val_loss: 0.7162
Epoch 2/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 966us/step - RootMeanSquaredError: 0.7356 - loss: 0.5417 - val_RootMeanSquaredError: 0.7640 - val_loss: 0.5837
Epoch 3/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 956us/step - RootMeanSquaredError: 0.6753 - loss: 0.4563 - val_RootMeanSquaredError: 1.0521 - val_loss: 1.1069
Epoch 4/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 939us/step - RootMeanSquaredError: 0.6489 - loss: 0.4212 - val_RootMeanSquaredError: 0.8224 - val_loss: 0.6763
Epoch 5/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 904us/step - RootMeanSquaredError: 0.6328 - loss: 0.4005 - val_RootMeanSquaredError: 1.5459 - val_loss: 2.3899
Epoch 6/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 986us/step - RootMeanSquaredError: 0.6219 - loss: 0.3869 - val_RootMeanSquaredError: 2.0893 - val_loss: 4.3653
Epoch 7/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - RootMeanSquaredError: 0.

Adding an auxiliary output for regularization:

In [13]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

In [14]:
input_wide = tf.keras.layers.Input(shape=[5])  # features 0 to 4
input_deep = tf.keras.layers.Input(shape=[6])  # features 2 to 7
norm_layer_wide = tf.keras.layers.Normalization()
norm_layer_deep = tf.keras.layers.Normalization()
norm_wide = norm_layer_wide(input_wide)
norm_deep = norm_layer_deep(input_deep)
hidden1 = tf.keras.layers.Dense(30, activation="relu")(norm_deep)
hidden2 = tf.keras.layers.Dense(30, activation="relu")(hidden1)
concat = tf.keras.layers.concatenate([norm_wide, hidden2])
output = tf.keras.layers.Dense(1)(concat)
aux_output = tf.keras.layers.Dense(1)(hidden2)
model = tf.keras.Model(inputs=[input_wide, input_deep], outputs=[output, aux_output])

In [15]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss=("mse", "mse"), loss_weights=(0.9, 0.1), optimizer=optimizer, metrics=["RootMeanSquaredError", "RootMeanSquaredError"])

In [16]:
norm_layer_wide.adapt(X_train_wide)
norm_layer_deep.adapt(X_train_deep)
history = model.fit(
    (X_train_wide, X_train_deep), (y_train, y_train), epochs=20,
    validation_data=((X_valid_wide, X_valid_deep), (y_valid, y_valid))
)

Epoch 1/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - dense_2_RootMeanSquaredError: 1.4237 - dense_2_loss: 2.1439 - dense_3_RootMeanSquaredError: 1.9498 - dense_3_loss: 3.9213 - loss: 2.3216 - val_dense_2_RootMeanSquaredError: 1.0711 - val_dense_2_loss: 1.1469 - val_dense_3_RootMeanSquaredError: 1.3241 - val_dense_3_loss: 1.7528 - val_loss: 1.2079
Epoch 2/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - dense_2_RootMeanSquaredError: 0.7089 - dense_2_loss: 0.5029 - dense_3_RootMeanSquaredError: 0.9254 - dense_3_loss: 0.8578 - loss: 0.5384 - val_dense_2_RootMeanSquaredError: 0.6420 - val_dense_2_loss: 0.4121 - val_dense_3_RootMeanSquaredError: 0.8548 - val_dense_3_loss: 0.7305 - val_loss: 0.4440
Epoch 3/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - dense_2_RootMeanSquaredError: 0.6663 - dense_2_loss: 0.4442 - dense_3_RootMeanSquaredError: 0.8002 - dense_3_loss: 0.6406 - loss: 0.4638 - val_dense_2_RootMeanSquaredError: 0.6221 - val_dense_2_loss: 0.3869 - val_dense_3_RootMeanSquaredError: 0.7

**Warning**: in recent TF version, ```evaluate()``` also returns the main metric and the aux metric. To ensure the code works in both old and new versions, we only look at the first 3 elements of ```eval_results``` (i.e., just the losses):

In [17]:
eval_results = model.evaluate((X_test_wide, X_test_deep), (y_test, y_test))
weighted_sum_of_losses, main_loss, aux_loss = eval_results[:3]

162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - dense_2_RootMeanSquaredError: 0.5784 - dense_2_loss: 0.3346 - dense_3_RootMeanSquaredError: 0.6274 - dense_3_loss: 0.3937 - loss: 0.3405  


In [18]:
y_pred_main, y_pred_aux = model.predict((X_new_wide, X_new_deep))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step


In [19]:
y_pred_tuple = model.predict((X_new_wide, X_new_deep))
y_pred = dict(zip(model.output_names, y_pred_tuple))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
